# J-Space Experiment — Phases 1 and 2

This is the canonical Colab launcher. Each phase runs in its own cell through the same CLI used locally or over SSH. Results are displayed between phases and saved under one run root. The notebook stores no scientific state outside that directory.

## 1. Install the experiment and pinned BIPIA checkout

In [ ]:
import subprocess
from pathlib import Path

RESEARCH_REPO = 'https://github.com/ethanncyb/jspace-research.git'
RESEARCH_REVISION = 'prompt-injection-experiment'
BIPIA_REVISION = 'a004b69ec0dd446e0afd461d98cb5e96e120a5d0'
REPO_ROOT = Path('/content/jspace-research')
BIPIA_CHECKOUT = Path('/content/BIPIA')

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', RESEARCH_REVISION, RESEARCH_REPO, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', RESEARCH_REVISION], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', RESEARCH_REVISION], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'merge', '--ff-only', f'origin/{RESEARCH_REVISION}'], check=True)
if not BIPIA_CHECKOUT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/microsoft/BIPIA.git', str(BIPIA_CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(BIPIA_CHECKOUT), 'checkout', BIPIA_REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-e', str(REPO_ROOT)], check=True)
print('Research revision:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('BIPIA revision:', subprocess.check_output(['git', '-C', str(BIPIA_CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip())

## 2. Authenticate

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import notebook_login

notebook_login()
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
print('OpenAI judge credential loaded from Colab Secrets.')

## 3. Configure one persistent run directory

In [ ]:
RUN_MODE = 'smoke'  # use 'full' only after smoke succeeds
USE_DRIVE = True
RUN_NAME = f'jspace-{RUN_MODE}'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/jspace-research/runs') / RUN_NAME
else:
    RUN_ROOT = Path('/content') / RUN_NAME

PHASE1_DIR = RUN_ROOT / 'phase1'
PHASE2_DIR = RUN_ROOT / 'phase2'
BIPIA_ROOT = BIPIA_CHECKOUT / 'benchmark'
CONFIG_PATH = REPO_ROOT / 'configs' / f'phase1_{RUN_MODE}.yaml'
WEBQA_TRAIN_PATH = None  # required for full mode
SUMMARIZATION_TRAIN_PATH = None  # required for full mode

def run_command(command, label):
    print('Running:', ' '.join(str(part) for part in command))
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{label} failed with exit status {return_code}; see the traceback above.')

print('Config:', CONFIG_PATH)
print('Run root:', RUN_ROOT)

## 4. Verify the GPU runtime

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before running the experiment.'
print('GPU:', torch.cuda.get_device_name(0))

## 5. Run or resume Phase 1

This freezes the manifest, captures activations, reconstructs J-space, and selects the layer. Rerunning the cell reuses compatible caches.

In [ ]:
phase1_command = [
    'jspace-phase1',
    '--config', str(CONFIG_PATH),
    '--bipia-root', str(BIPIA_ROOT),
    '--output-dir', str(PHASE1_DIR),
    '--stage', 'all',
]
if WEBQA_TRAIN_PATH is not None:
    phase1_command.extend(['--webqa-train', str(WEBQA_TRAIN_PATH)])
if SUMMARIZATION_TRAIN_PATH is not None:
    phase1_command.extend(['--summarization-train', str(SUMMARIZATION_TRAIN_PATH)])
run_command(phase1_command, 'Phase 1')

## 6. Inspect Phase 1 before continuing

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

selection = json.loads((PHASE1_DIR / 'selected_layer.json').read_text())
print(json.dumps(selection, indent=2))
display(pd.read_csv(PHASE1_DIR / 'layer_metrics.csv'))
display(Image(filename=str(PHASE1_DIR / 'layer_auprc.png')))
display(Image(filename=str(PHASE1_DIR / 'selected_layer_score_distribution.png')))

## 7. Run or resume Phase 2 generation

This GPU stage reads the frozen Phase 1 directory directly and runs three conditions: intact (`alpha=0.0`), partial removal (`alpha=0.5`), and full removal (`alpha=1.0`). The selected artifact path below is the only notebook-level connection between phases.

In [ ]:
phase2_base = [
    'jspace-phase2',
    '--config', str(CONFIG_PATH),
    '--phase1', str(PHASE1_DIR / 'selected_layer.json'),
    '--output-dir', str(PHASE2_DIR),
]
run_command([*phase2_base, '--stage', 'generate'], 'Phase 2 generation')

## 8. Run or resume Phase 2 analysis

This stage uses cached generations, ROUGE scoring, and the pinned API judge. It can also be run later on a CPU machine after copying the complete run directory.

In [ ]:
run_command([*phase2_base, '--stage', 'analyze'], 'Phase 2 analysis')

## 9. Inspect Phase 2 results

In [ ]:
display(pd.read_csv(PHASE2_DIR / 'phase2_summary.csv'))
display(pd.read_csv(PHASE2_DIR / 'phase2_examples.csv'))
display(Image(filename=str(PHASE2_DIR / 'phase2_asr_vs_alpha.png')))
display(Image(filename=str(PHASE2_DIR / 'phase2_clean_utility_vs_alpha.png')))

## 10. Confirm persistence

With `USE_DRIVE = True`, all caches and results are already saved in Drive. With ephemeral `/content` storage, copy or download the entire run root before the Colab runtime ends. Preserve the `phase1/` and `phase2/` directories together.

In [ ]:
print('Complete run root:', RUN_ROOT)
print('Phase 1 selected layer:', selection['selected_layer'])
print('Phase 2 results:', PHASE2_DIR / 'phase2_results.parquet')
assert (PHASE1_DIR / 'selected_layer.json').is_file()
assert (PHASE2_DIR / 'phase2_results.parquet').is_file()

## Interpretation boundary

An effect in Phase 2 shows that the selected layer's reconstructed J-space component is functionally involved in model behavior. It does not establish injection-specific causality.